# CS336 Assignment 2 (Systems) 实验方案与记录

> 硬件：服务器 2 × RTX 5070 Ti（Blackwell, sm_120, 16GB GDDR7，无 NVLink）
> 说明：本方案基于 Spring 2025 版作业结构整理（Profiling / FlashAttention-2 / DDP / 优化器状态分片）。表格可按实际题目增减。

---

## 0. 总原则

1. **先测量，再优化，再解释**：每个实验都要有基线、数字、因果解释。
2. **正确性先于性能**：每个并行或 kernel 版本，先和单卡 / 参考实现对齐，再测速度。
3. **降配要透明**：凡偏离作业设定（模型规模、context、batch），在报告里写明原因，并保证"相对对比"仍然成立。
4. **OOM 也是结果**：记录 OOM 的配置和报错，并给出理论显存估算。
5. **每组实验记录环境**（见第 1 节），否则数字不可复现。

---



## 1. 环境准备

### 1.1 检查清单

- [-] `nvidia-smi` 能看到 2 张卡，且没有其他任务占用（共享服务器务必确认）
- [-] `nvidia-smi topo -m` 查看两卡互联方式（PHB / PXB / SYS 等）
- [-] PyTorch 为 cu128 及以上构建（支持 sm_120）；Triton 3.3+
- [-] `torch.cuda.get_device_capability()` 返回 (12, 0)
- [-] nsys 可用（`nsys --version`）；ncu 可用则更好
- [-] 作业仓库依赖装完，`pytest` 能跑起来（测试会先全部失败，这是正常的）
- [-] 固定随机种子，关闭 TF32 与否要统一并记录

---


### 1.2 环境记录表（每次实验批次附一份）

| 项目 | 值 |
|---|---|
| 日期 | 26-1016 |
| GPU 型号 / 驱动版本 | 2× NVIDIA GeForce RTX 5070 Ti (16 GB), driver 580.178.04 |
| CUDA / cuDNN / NCCL 版本 | CUDA Version: 13.0 |
| PyTorch / Triton 版本 | vllm(2.8.0+cu128/3.4.0) ,nv(2.11.0/3.6.0)|
| 两卡互联（topo -m 结果） | Gen4 x16 per slot; both GPUs under same PHB; single NUMA node |
| P2P 是否可用 |disabled (CNS — chipset not supported)  |
| NCCL | SHM/direct/direct transport, 2 channels |
| GPU 频率 / 功耗限制 |  300W |
| 代码 commit | cs336自己仓库，参考third-party |

### 1.3 通信基线：all_reduce 带宽测试

作用：Part 3 所有通信开销结论的引用依据。

**环境**：2× RTX 5070 Ti (16GB)，PCIe Gen4 x16，PHB/单 NUMA/无 NVLink，
P2P 不可用（topo → CNS）；NCCL 默认走 **SHM/direct/direct**。

**主表**（默认 SHM，back-to-back 计时）：

| tensor 大小 | 迭代 | 延迟 (ms) | algbw (GB/s) | busbw (GB/s) | 备注 |
|---|---|---|---|---|---|
| 1 MB   | 200 | 0.203   | 5.17 | 5.17 | 无可见延迟开销 |
| 4 MB   | 100 | 0.791   | 5.30 | 5.30 | |
| 16 MB  | 50  | 3.213   | 5.22 | 5.22 | |
| 64 MB  | 20  | 12.947  | 5.18 | 5.18 | |
| 256 MB | 10  | 51.374  | 5.23 | 5.23 | |
| 1 GB   | 5   | 206.827 | 5.19 | 5.19 | 平台区 |

**对照组**（`NCCL_SHM_DISABLE=1` → NET/socket，NCCL_DEBUG 确认）：
64 MB → 39.9 ms / 1.68 GB/s；256 MB → 163.0 ms / 1.65 GB/s（**慢 3.2×**）

**结论**
1. 平台值 ≈ **5.2 GB/s**，1 MB–1 GB 全程平坦（极差 <3%）——≥1 MB 无延迟地板
   （back-to-back 流水线掩盖每次调用固定开销；地板只可能出现在 sub-MB 桶）
2. NCCL 传输选型影响重大：SHM→socket 损失 3.2×
3. fp32 梯度同步心算（÷5.2 GB/s）：30M→23ms，124M→95ms，500M→394ms
4. DDP 分桶的代价锚点是 **CPU launch 开销 + overlap 破坏**，不是 NCCL 延迟（本机实测）


---

## 2. 模型配置参考与显存预算

### 2.1 作业模型规格（vocab = 10,000，batch size = 4）

| 模型 | d_model | d_ff | layers | heads | 参数量（约） | fp32+AdamW 训练静态显存（约 16 B/param） |
|---|---|---|---|---|---|---|
| small | 768 | 3072 | 12 | 12 | 0.13B | 2 GB |
| medium | 1024 | 4096 | 24 | 16 | 0.35B | 6 GB |
| large | 1280 | 5120 | 36 | 20 | 0.76B | 12 GB（激活后很紧） |
| xl | 1600 | 6400 | 48 | 25 | 1.6B | 26 GB（16GB 卡必 OOM） |
| 2.7B | 2560 | 10240 | 32 | 32 | 2.7B | 43 GB（必 OOM） |

### 2.2 降配策略总表

| 实验 | 作业默认 | 本机方案 |
|---|---|---|
| Part 1 benchmark | small ~ 2.7B | small / medium 全做；large 缩 context；xl、2.7B 仅 forward（bf16），完整训练步记录 OOM |
| Part 1 显存 profiling | 2.7B | medium 或 large 代替，同时给出 2.7B 的理论估算 |
| Part 2 FlashAttention | 无模型限制 | 原样 |
| Part 3 DDP | xl，2 GPU | medium / large，2 GPU |
| Part 4 优化器分片 | xl，2 GPU | large（小 batch）或 medium |

#### 官方源码方案：subtree
- 主目录uv建了官方仓库依赖一样的环境
- conda 的环境用于notebook测试
- Git Subtree
```bash
cd my-llm-sys
git subtree add --prefix=third_party/assignment2-systems \
    https://github.com/stanford-cs336/assignment2-systems.git main --squash
--squash 会把上游所有历史压缩成一个 commit，不污染git log。
```
拉取上游更新：

```bash
git fetch https://github.com/stanford-cs336/assignment2-systems.git main
git subtree pull --prefix=third_party/assignment2-systems \
    https://github.com/stanford-cs336/assignment2-systems.git main --squash
```

### 测试记录

In [9]:
import subprocess, shutil, textwrap, json, sys

def sh(cmd: list[str]) -> str:
    """运行系统命令并返回 stdout；命令不存在时返回友好提示"""
    if shutil.which(cmd[0]) is None:
        return f"[MISSING] {cmd[0]} not found in PATH"
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    return (r.stdout + r.stderr).strip()

# 1.1 列出所有 GPU + 当前占用进程
print(sh(["nvidia-smi"]))

Mon Oct 26 16:28:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     Off |   00000000:2D:00.0  On |                  N/A |
|  0%   36C    P8              6W /  300W |    1166MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

- 两卡互联

In [10]:
print(sh(["nvidia-smi", "topo", "-m"]))
import torch

print("P2P 0→1:", torch.cuda.can_device_access_peer(0, 1))
print("P2P 1→0:", torch.cuda.can_device_access_peer(1, 0))

a = torch.empty(256 * 1024 * 1024, dtype=torch.uint8, device="cuda:0")  # 256 MB
b = torch.empty_like(a, device="cuda:1")
for _ in range(3): b.copy_(a)                      # warmup
torch.cuda.synchronize()
s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
s.record()
for _ in range(10): b.copy_(a, non_blocking=True)
e.record(); torch.cuda.synchronize()
t = s.elapsed_time(e) / 10 / 1000                  # 秒
print(f"GPU0→GPU1 拷贝带宽: {a.numel()/t/1e9:.1f} GB/s")


GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-31	0		N/A
GPU1	PHB	 X 	0-31	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
P2P 0→1: False
P2P 1→0: False
GPU0→GPU1 拷贝带宽: 11.8 GB/s


In [17]:
print(sh(["nvidia-smi","--query-gpu=index,name,memory.total,driver_version,pcie.link.gen.max,pcie.link.width.max","--format=csv"]))  
print(sh(["nvidia-smi","topo","-p2p","r"])) 

index, name, memory.total [MiB], driver_version, pcie.link.gen.max, pcie.link.width.max
0, NVIDIA GeForce RTX 5070 Ti, 16303 MiB, 580.178.04, 4, 16
1, NVIDIA GeForce RTX 5070 Ti, 16303 MiB, 580.178.04, 4, 16
GPU0	GPU1	
 GPU0	X	CNS	
 GPU1	CNS	X	

Legend:

  X    = Self
  OK   = Status Ok
  CNS  = Chipset not supported
  GNS  = GPU not supported
  TNS  = Topology not supported
  NS   = Not supported
  U    = Unknown


- torch&trition version

In [11]:
import torch

print(f"torch           : {torch.__version__}")
print(f"torch.cuda avail: {torch.cuda.is_available()}")
print(f"torch.version.cuda (build): {torch.version.cuda}")
print(f"cudnn.version   : {torch.backends.cudnn.version()}")
print(f"device count    : {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cap = torch.cuda.get_device_capability(i)
    print(f"[cuda:{i}] {p.name} | capability sm_{cap[0]}{cap[1]} | "
          f"{p.total_memory/1024**3:.1f} GB | CC {p.major}.{p.minor}")

# 关键校验：sm_120 必须出现在 build 支持列表里
supported = torch.cuda.get_arch_list()
print(f"arch list       : {supported}")
assert any("120" in a for a in supported), \
    "❌ 当前 torch wheel 不含 sm_120，请安装 cu128/cu129 版本"
print("✅ sm_120 支持 OK")


torch           : 2.8.0+cu128
torch.cuda avail: True
torch.version.cuda (build): 12.8
cudnn.version   : 91002
device count    : 2
[cuda:0] NVIDIA GeForce RTX 5070 Ti | capability sm_120 | 15.5 GB | CC 12.0
[cuda:1] NVIDIA GeForce RTX 5070 Ti | capability sm_120 | 15.5 GB | CC 12.0
arch list       : ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
✅ sm_120 支持 OK


In [12]:
import triton
import triton.language as tl

print(f"triton.__version__ : {triton.__version__}")
print(f"triton path        : {triton.__file__}")

# 最小可编译 kernel：一个向量加法
@triton.jit
def _add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    idx = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = idx < n
    x = tl.load(x_ptr + idx, mask=mask)
    y = tl.load(y_ptr + idx, mask=mask)
    tl.store(out_ptr + idx, x + y, mask=mask)

x = torch.randn(4096, device="cuda")
y = torch.randn(4096, device="cuda")
out = torch.empty_like(x)
_add_kernel[(1,)](x, y, out, 4096, BLOCK=4096)

assert torch.allclose(out, x + y), "❌ Triton kernel 结果错误"
print("✅ Triton 在 sm_120 上编译 & 运行 OK")


triton.__version__ : 3.4.0
triton path        : /data2/conda/anaconda3/envs/vllm/lib/python3.11/site-packages/triton/__init__.py
✅ Triton 在 sm_120 上编译 & 运行 OK


- nsys&ncu

In [13]:
print("nsys:", sh(["nsys", "--version"]))
print("ncu :", sh(["ncu", "--version"]))

nsys: NVIDIA Nsight Systems version 2024.6.2.225-246235244400v0
ncu : NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2024 NVIDIA Corporation
Version 2025.1.0.0 (build 35237751) (public-release)


- uv

In [14]:
import subprocess

REPO = "/data2/c220/hz/cs336"
CHECK = """
import sys, torch, triton
print(f"python : {sys.version.split()[0]}")
print(f"torch  : {torch.__version__}  (cuda build: {torch.version.cuda})")
print(f"triton : {triton.__version__}")
print(f"arch   : {torch.cuda.get_arch_list()}")
assert torch.cuda.is_available(), "❌ CUDA 不可用"
assert any("120" in a for a in torch.cuda.get_arch_list()), "❌ 不含 sm_120"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU    : {p.name} | sm_{cap[0]}{cap[1]} | {p.total_memory/1024**3:.1f} GB")
print("✅ uv 环境检查通过")
"""

r = subprocess.run(["uv", "run", "python", "-c", CHECK],
                   cwd=REPO, capture_output=True, text=True, timeout=120)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-1500:])


python : 3.13.17
torch  : 2.11.0+cu130  (cuda build: 13.0)
triton : 3.6.0
arch   : ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU    : NVIDIA GeForce RTX 5070 Ti | sm_120 | 15.5 GB
✅ uv 环境检查通过



### pytest环境

- pytest 起跑验证
官方 tests 依赖官方仓库根的 pyproject.toml 作为 rootdir 来解析 cs336_basics / cs336_systems 的 import，而用 subtree 后这个 pyproject 在 third_party/assignment2-systems/ 下。
所以正确做法是 cwd 指到 subtree 里再调 pytest，让 uv 用官方 pyproject 解析环境（第一次会花几分钟下载依赖，之后有缓存）。
- REPO    = "/data2/c220/hz/cs336"                          # 仓库根（策略 2 主战场）

In [15]:
import os, re, subprocess

# ========== 配置区 ==========
REPO    = "/data2/c220/hz/cs336"                          # 仓库根（策略 2 主战场）
SUBTREE = f"{REPO}/third_party/assignment2-systems"       # 官方原版，只读对照

# 防御：kernel 启动早于 .bashrc，subprocess 继承不到，手动补
os.environ.setdefault("UV_CACHE_DIR",     "/data2/c220/hz/.uv-cache")
os.environ.setdefault("TRITON_CACHE_DIR", "/data2/c220/hz/.triton-cache")
os.environ.setdefault("HF_HOME",          "/data2/c220/hz/.hf-cache")
# ============================

# 1. 主仓库项目形态检查（策略 2 终态）
print("─" * 60)
print("1) Repo structure (strategy-2 layout)")
paths = {
    "pyproject (mine)":   (f"{REPO}/pyproject.toml",                       True),
    "uv.lock":            (f"{REPO}/uv.lock",                              True),
    ".venv python":       (f"{REPO}/.venv/bin/python",                     True),
    "cs336_systems":      (f"{REPO}/cs336_systems/__init__.py",            True),
    "tests/ (copied)":    (f"{REPO}/tests",                                True),
    "tests/adapters.py":  (f"{REPO}/tests/adapters.py",                    True),
    "tests/common.py":    (f"{REPO}/tests/common.py",                      True),
    "subtree (readonly)": (f"{SUBTREE}/pyproject.toml",                    True),
}
ALL_OK = True
for name, (p, req) in paths.items():
    ok = os.path.exists(p)
    ALL_OK &= (ok or not req)
    print(f"  {'✅' if ok else '❌'} {name:20s} → {p}")
print("─" * 60)

# 2. Editable 校验：import 解析到的必须是"你的"源码
print("2) Editable import check")
r = subprocess.run(
    ["uv", "run", "python", "-c",
     "import cs336_systems, cs336_basics; "
     "print(cs336_systems.__file__); print(cs336_basics.__file__)"],
    cwd=REPO, capture_output=True, text=True, timeout=120,
)
print(r.stdout.strip() or r.stderr.strip()[-800:])
EDITABLE_OK = (r.returncode == 0
               and f"{REPO}/cs336_systems" in r.stdout
               and "third_party/assignment2-systems/cs336-basics" in r.stdout)
print(f"  editable: {'✅' if EDITABLE_OK else '❌ (import 的是错的包!)'}")
print("─" * 60)

# 3. collect-only（rootdir = 主仓库）
print("3) pytest --collect-only (rootdir = REPO)")
result = subprocess.run(
    ["uv", "run", "pytest", "tests", "--collect-only", "-q"],
    cwd=REPO, capture_output=True, text=True, timeout=600,
)
print(result.stdout[-4000:])
if result.returncode != 0 and not result.stdout.strip():
    print("STDERR:", result.stderr[-2000:])
m = re.search(r"(\d+) tests? collected", result.stdout)
PYTEST_COUNT = int(m.group(1)) if m else 0
PYTEST_OK = result.returncode == 0 and PYTEST_COUNT > 0
print("─" * 60)

# 4) Smoke（修正版）：用 A2 真实测试，预期 FAILED + NotImplementedError
print("4) Smoke: test_flash_forward_pass_pytorch")
smoke = subprocess.run(
    ["uv", "run", "pytest", "tests/test_attention.py",
     "-k", "test_flash_forward_pass_pytorch", "-x"],
    cwd=REPO, capture_output=True, text=True, timeout=300,
)
print(smoke.stdout[-2500:])
if "NotImplementedError" not in smoke.stdout and smoke.returncode != 0:
    print("STDERR:", smoke.stderr[-800:])

SMOKE_OK = ("NotImplementedError" in smoke.stdout
            and "ImportError" not in smoke.stdout
            and "ModuleNotFoundError" not in smoke.stdout)
print("─" * 60)
print("smoke:", "✅ 测试跑进了你的空实现，抛 NotImplementedError（完全符合预期）"
      if SMOKE_OK else "❌ 看上面输出排查")


# 5. 总判决
print("VERDICT")
print(f"  structure : {'✅' if ALL_OK        else '❌'}")
print(f"  editable  : {'✅' if EDITABLE_OK  else '❌'}")
print(f"  collected : {PYTEST_COUNT:>4}     {'✅' if PYTEST_OK else '❌'}")
print(f"  smoke     : {'✅ (fail=预期,待填 adapters)' if NO_IMPORT_ERR else '❌ import 级错误,先修环境'}")
READY = ALL_OK and EDITABLE_O

────────────────────────────────────────────────────────────
1) Repo structure (strategy-2 layout)
  ✅ pyproject (mine)     → /data2/c220/hz/cs336/pyproject.toml
  ✅ uv.lock              → /data2/c220/hz/cs336/uv.lock
  ✅ .venv python         → /data2/c220/hz/cs336/.venv/bin/python
  ✅ cs336_systems        → /data2/c220/hz/cs336/cs336_systems/__init__.py
  ✅ tests/ (copied)      → /data2/c220/hz/cs336/tests
  ✅ tests/adapters.py    → /data2/c220/hz/cs336/tests/adapters.py
  ✅ tests/common.py      → /data2/c220/hz/cs336/tests/common.py
  ✅ subtree (readonly)   → /data2/c220/hz/cs336/third_party/assignment2-systems/pyproject.toml
────────────────────────────────────────────────────────────
2) Editable import check
/data2/c220/hz/cs336/cs336_systems/__init__.py
/data2/c220/hz/cs336/third_party/assignment2-systems/cs336-basics/cs336_basics/__init__.py
  editable: ✅
────────────────────────────────────────────────────────────
3) pytest --collect-only (rootdir = REPO)
tests/test_attention.py

NameError: name 'EDITABLE_O' is not defined

- seed

In [ ]:
import random, numpy as np, torch

def set_repro(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # benchmark 模式：形状固定时 cudnn 自动选最快算法（A2 benchmark 要速度）
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print(f"repro seed = {seed}")

set_repro()


repro seed = 42
